# Gupio Option 2 — Bank Term Deposit Subscription Prediction

**Goal:** Predict whether a client will subscribe to a bank term deposit using information available immediately before the current contact begins.

This notebook follows the Gupio brief:
- semicolon-delimited `bank-full.csv`
- fixed random seed
- stratified train/test split
- leakage-safe preprocessing
- at least two binary classifiers
- Accuracy, Precision, Recall, F1, Confusion Matrix and ROC-AUC
- discussion of class imbalance and business trade-offs
- final model selection based on evaluation evidence
- sample predictions using the same preprocessing pipeline

**Important leakage rule:** `contact`, `day`, `month`, `duration`, and `campaign` are excluded from the primary model.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

RANDOM_STATE = 42
DATA_PATH = Path("../data/bank-full.csv")

df = pd.read_csv(DATA_PATH, sep=";")
df.shape


: 

In [ ]:
# Basic inspection
print("Shape:", df.shape)
display(df.head())
display(df.dtypes)
print("\nMissing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("\nTarget distribution:")
display(df["y"].value_counts())
display(df["y"].value_counts(normalize=True).mul(100).round(2))


## Prediction-time leakage prevention

The Gupio brief says prediction occurs immediately before the current campaign contact. Therefore the primary model excludes:
- `contact`
- `day`
- `month`
- `duration`
- `campaign`

In particular, `duration` is only known after the call and would leak information from the contact being predicted.


In [ ]:
LEAKAGE_COLS = ["contact", "day", "month", "duration", "campaign"]
X = df.drop(columns=LEAKAGE_COLS + ["y"]).copy()
y = (df["y"] == "yes").astype(int)

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)


In [ ]:
# Target distribution
counts = df["y"].value_counts().reindex(["no","yes"])
counts.plot(kind="bar")
plt.title("Target Distribution")
plt.xlabel("Subscription (y)")
plt.ylabel("Number of clients")
plt.xticks(rotation=0)
plt.show()

# Subscription rate by job
job_rate = df.groupby("job")["y"].apply(lambda s: (s=="yes").mean()).sort_values(ascending=False)
job_rate.plot(kind="bar", figsize=(9,5))
plt.title("Subscription Rate by Job")
plt.ylabel("Subscription rate")
plt.xticks(rotation=45, ha="right")
plt.show()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

def make_preprocessor(scale_numeric=True):
    return ColumnTransformer([
        ("num", StandardScaler() if scale_numeric else "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ])

models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", make_preprocessor(True)),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", make_preprocessor(False)),
        ("model", RandomForestClassifier(
            n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1
        ))
    ]),
    "Logistic Regression (Balanced)": Pipeline([
        ("preprocessor", make_preprocessor(True)),
        ("model", LogisticRegression(
            max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced"
        ))
    ]),
}


In [ ]:
rows = []
fitted = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    cm = confusion_matrix(y_test, pred)
    fitted[name] = model
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, proba),
        "TN": cm[0,0], "FP": cm[0,1], "FN": cm[1,0], "TP": cm[1,1],
    })

results = pd.DataFrame(rows)
display(results.round(4))


## Model selection

The positive class is `yes`. The target is imbalanced, so accuracy alone can be misleading.

For a campaign-targeting use case, recall is important because a false negative means missing a client who would subscribe. Precision also matters because false positives consume outreach resources.

The balanced Logistic Regression model is selected for the submitted baseline because it gives the strongest positive-class F1 among the tested models while substantially improving recall compared with the unweighted baseline. This is a modelling choice for prediction; it does **not** imply causal conclusions about customer behaviour.


In [ ]:
final_model = fitted["Logistic Regression (Balanced)"]
final_pred = final_model.predict(X_test)

ConfusionMatrixDisplay.from_predictions(
    y_test, final_pred, display_labels=["no", "yes"], values_format="d"
)
plt.title("Final Model - Confusion Matrix")
plt.show()

print(classification_report(
    y_test, final_pred, target_names=["no", "yes"], digits=4, zero_division=0
))


In [ ]:
# Sample predictions
sample = X_test.head(5).copy()
sample["predicted_subscription"] = np.where(final_model.predict(sample)==1, "yes", "no")
sample["probability_yes"] = final_model.predict_proba(X_test.head(5))[:,1]
display(sample)


## Limitations

1. The dataset represents a historical marketing campaign and may not generalize to another bank, time period or customer population.
2. The model learns predictive associations, not causal effects.
3. Class imbalance creates a precision/recall trade-off.
4. The primary model deliberately excludes current-contact information to respect the stated prediction point.
5. A future production system should monitor drift, calibration, fairness and business costs and should validate performance prospectively.
